In [2]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
import os

np.random.seed(42)
random.seed(42)

N = 50000  # Total transactions
FRAUD_RATE = 0.08  # 8% fraud

print("🔧 Generating synthetic e-commerce fraud dataset...")

# --- Time features ---
start_date = datetime(2023, 1, 1)
timestamps = [start_date + timedelta(minutes=random.randint(0, 525600)) for _ in range(N)]
timestamps.sort()

# --- User & product features ---
user_ids     = [f"USR{random.randint(1000, 9999)}" for _ in range(N)]
product_cats = np.random.choice(["Electronics","Fashion","Grocery","Beauty","Sports","Books"], N,
                                 p=[0.25, 0.20, 0.20, 0.15, 0.12, 0.08])
devices      = np.random.choice(["Mobile","Desktop","Tablet"], N, p=[0.55, 0.35, 0.10])
browsers     = np.random.choice(["Chrome","Safari","Firefox","Edge","Unknown"], N,
                                 p=[0.40, 0.25, 0.15, 0.10, 0.10])
countries    = np.random.choice(["India","USA","UK","Germany","Nigeria","Russia","China","Brazil"], N,
                                 p=[0.30, 0.25, 0.15, 0.08, 0.07, 0.06, 0.05, 0.04])
payment_methods = np.random.choice(["Credit Card","Debit Card","UPI","Net Banking","Wallet"], N,
                                    p=[0.30, 0.25, 0.20, 0.15, 0.10])
merchants    = [f"MERCH{random.randint(100, 300)}" for _ in range(N)]

# --- Amount (fraud txns tend to be higher or oddly specific) ---
amounts = np.where(
    np.random.rand(N) < FRAUD_RATE,
    np.round(np.random.exponential(scale=4500, size=N), 2),   # fraud: higher amounts
    np.round(np.random.exponential(scale=1200, size=N), 2)    # legit: lower amounts
)
amounts = np.clip(amounts, 50, 50000)

# --- IP features ---
ip_country_match = np.random.choice([True, False], N, p=[0.85, 0.15])

# --- Account age (days) ---
account_age_days = np.random.randint(1, 2000, N)

# --- Velocity: orders in last 1 hour by same user (higher = suspicious) ---
velocity_1h = np.random.choice([1, 2, 3, 4, 5, 6, 7, 8], N,
                                p=[0.55, 0.20, 0.10, 0.06, 0.04, 0.02, 0.02, 0.01])

# --- Previous chargebacks ---
prev_chargebacks = np.random.choice([0, 1, 2, 3], N, p=[0.88, 0.08, 0.03, 0.01])

# --- Device fingerprint mismatch ---
device_mismatch = np.random.choice([False, True], N, p=[0.90, 0.10])

# --- Shipping != billing address ---
address_mismatch = np.random.choice([False, True], N, p=[0.80, 0.20])

# --- Failed attempts before success ---
failed_attempts = np.random.choice([0, 1, 2, 3, 4], N, p=[0.75, 0.12, 0.07, 0.04, 0.02])

# ----------------------------------------------------------------
# FRAUD LABEL — rule-based with noise to simulate real patterns
# ----------------------------------------------------------------
fraud_score_raw = (
    (amounts > 5000).astype(int) * 2 +
    (~ip_country_match).astype(int) * 2 +
    (velocity_1h >= 4).astype(int) * 2 +
    (prev_chargebacks >= 1).astype(int) * 3 +
    (device_mismatch).astype(int) * 1 +
    (address_mismatch).astype(int) * 1 +
    (failed_attempts >= 2).astype(int) * 2 +
    (account_age_days < 30).astype(int) * 1 +
    (browsers == "Unknown").astype(int) * 1 +
    (countries.isin(["Nigeria", "Russia"]) if isinstance(countries, pd.Series)
     else np.isin(countries, ["Nigeria", "Russia"])).astype(int) * 2
)

fraud_prob = fraud_score_raw / fraud_score_raw.max()
fraud_label = (fraud_prob > np.percentile(fraud_prob, 100 - FRAUD_RATE * 100)).astype(int)
# Add noise
noise_idx = np.random.choice(N, size=int(N * 0.01), replace=False)
fraud_label[noise_idx] = 1 - fraud_label[noise_idx]

# ----------------------------------------------------------------
# Assemble DataFrame
# ----------------------------------------------------------------
df = pd.DataFrame({
    "transaction_id":    [f"TXN{100000+i}" for i in range(N)],
    "timestamp":         timestamps,
    "user_id":           user_ids,
    "merchant_id":       merchants,
    "amount":            amounts,
    "product_category":  product_cats,
    "payment_method":    payment_methods,
    "device_type":       devices,
    "browser":           browsers,
    "country":           countries,
    "ip_country_match":  ip_country_match,
    "account_age_days":  account_age_days,
    "velocity_1h":       velocity_1h,
    "prev_chargebacks":  prev_chargebacks,
    "device_mismatch":   device_mismatch,
    "address_mismatch":  address_mismatch,
    "failed_attempts":   failed_attempts,
    "is_fraud":          fraud_label
})

df["hour"]       = df["timestamp"].dt.hour
df["day_of_week"] = df["timestamp"].dt.day_name()
df["month"]      = df["timestamp"].dt.month

os.makedirs("data", exist_ok=True)
df.to_csv("data/transactions.csv", index=False)

print(f"✅ Dataset saved → data/transactions.csv")
print(f"   Total transactions : {N:,}")
print(f"   Fraud transactions : {fraud_label.sum():,} ({fraud_label.mean()*100:.1f}%)")
print(f"   Legit transactions : {(1-fraud_label).sum():,}")
print(df.head(3).to_string())

🔧 Generating synthetic e-commerce fraud dataset...
✅ Dataset saved → data/transactions.csv
   Total transactions : 50,000
   Fraud transactions : 2,775 (5.5%)
   Legit transactions : 47,225
  transaction_id           timestamp  user_id merchant_id  amount product_category payment_method device_type browser  country  ip_country_match  account_age_days  velocity_1h  prev_chargebacks  device_mismatch  address_mismatch  failed_attempts  is_fraud  hour day_of_week  month
0      TXN100000 2023-01-01 00:09:00  USR1760    MERCH212  440.95          Fashion    Credit Card     Desktop  Safari    India             False               912            1                 2            False             False                3         1     0      Sunday      1
1      TXN100001 2023-01-01 00:37:00  USR6323    MERCH250  451.72            Books     Debit Card      Mobile  Safari       UK              True               190            3                 0            False             False                1   

In [3]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import os, warnings
warnings.filterwarnings("ignore")

os.makedirs("outputs/plots", exist_ok=True)

df = pd.read_csv("data/transactions.csv", parse_dates=["timestamp"])
fraud  = df[df["is_fraud"] == 1]
legit  = df[df["is_fraud"] == 0]

palette = {"Fraud": "#E63946", "Legit": "#457B9D"}
sns.set_theme(style="whitegrid", font_scale=1.1)

print("📊 Running Exploratory Data Analysis...")

# ─────────────────────────────────────────────────────────────
# PLOT 1 ─ Overview Dashboard (2x3 grid)
# ─────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(20, 14))
fig.patch.set_facecolor("#F8F9FA")
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# 1a ─ Fraud vs Legit donut
ax1 = fig.add_subplot(gs[0, 0])
sizes  = [len(legit), len(fraud)]
colors = ["#457B9D", "#E63946"]
wedges, texts, autotexts = ax1.pie(sizes, labels=["Legit","Fraud"],
                                    autopct="%1.1f%%", colors=colors,
                                    startangle=90, pctdistance=0.75,
                                    wedgeprops=dict(width=0.55, edgecolor="white", linewidth=2))
for at in autotexts: at.set_fontsize(13); at.set_fontweight("bold")
ax1.set_title("Overall Fraud Rate", fontsize=14, fontweight="bold", pad=12)

# 1b ─ Amount distribution
ax2 = fig.add_subplot(gs[0, 1])
ax2.hist(legit["amount"].clip(0, 10000),  bins=50, alpha=0.7, color="#457B9D", label="Legit",  density=True)
ax2.hist(fraud["amount"].clip(0, 10000),  bins=50, alpha=0.7, color="#E63946", label="Fraud",  density=True)
ax2.set_xlabel("Transaction Amount (₹)"); ax2.set_ylabel("Density")
ax2.set_title("Amount Distribution", fontsize=14, fontweight="bold")
ax2.legend(); ax2.xaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f"₹{x:,.0f}"))

# 1c ─ Fraud rate by product category
ax3 = fig.add_subplot(gs[0, 2])
cat_fraud = df.groupby("product_category")["is_fraud"].mean().sort_values(ascending=False) * 100
bars = ax3.barh(cat_fraud.index, cat_fraud.values,
                color=["#E63946" if v > cat_fraud.mean() else "#457B9D" for v in cat_fraud.values],
                edgecolor="white", linewidth=0.8)
for bar, val in zip(bars, cat_fraud.values):
    ax3.text(val + 0.1, bar.get_y() + bar.get_height()/2, f"{val:.1f}%", va="center", fontsize=11)
ax3.set_xlabel("Fraud Rate (%)"); ax3.set_title("Fraud Rate by Category", fontsize=14, fontweight="bold")

# 1d ─ Hourly fraud heatmap by day
ax4 = fig.add_subplot(gs[1, :2])
day_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
heat_data = df[df["is_fraud"]==1].groupby(["day_of_week","hour"]).size().unstack(fill_value=0)
heat_data = heat_data.reindex([d for d in day_order if d in heat_data.index])
sns.heatmap(heat_data, ax=ax4, cmap="YlOrRd", linewidths=0.3,
            cbar_kws={"label":"Fraud Count"}, annot=False)
ax4.set_title("🔥 Fraud Heatmap — Day of Week vs Hour", fontsize=14, fontweight="bold")
ax4.set_xlabel("Hour of Day"); ax4.set_ylabel("")

# 1e ─ Fraud by country
ax5 = fig.add_subplot(gs[1, 2])
country_fraud = df.groupby("country")["is_fraud"].mean().sort_values(ascending=False) * 100
colors_c = ["#E63946" if v > country_fraud.mean() else "#457B9D" for v in country_fraud.values]
ax5.barh(country_fraud.index, country_fraud.values, color=colors_c, edgecolor="white")
ax5.set_xlabel("Fraud Rate (%)"); ax5.set_title("Fraud Rate by Country", fontsize=14, fontweight="bold")
for i, v in enumerate(country_fraud.values):
    ax5.text(v + 0.1, i, f"{v:.1f}%", va="center", fontsize=10)

plt.suptitle("E-Commerce Fraud Detection — EDA Overview", fontsize=18,
             fontweight="bold", y=1.01, color="#1D3557")
fig.savefig("outputs/plots/01_eda_overview.png", dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.close()
print("   ✅ Plot 1 saved: 01_eda_overview.png")

# ─────────────────────────────────────────────────────────────
# PLOT 2 ─ Behavioral Risk Signals
# ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.patch.set_facecolor("#F8F9FA")
fig.suptitle("Behavioral Risk Signals Analysis", fontsize=18, fontweight="bold", color="#1D3557", y=1.01)

# 2a ─ Velocity vs fraud rate
ax = axes[0, 0]
vel_fraud = df.groupby("velocity_1h")["is_fraud"].mean() * 100
ax.bar(vel_fraud.index, vel_fraud.values,
       color=["#E63946" if v > vel_fraud.mean() else "#2A9D8F" for v in vel_fraud.values],
       edgecolor="white", width=0.7)
ax.set_xlabel("Orders in Last 1 Hour"); ax.set_ylabel("Fraud Rate (%)")
ax.set_title("Velocity vs Fraud Rate", fontweight="bold")
for i, v in enumerate(vel_fraud.values):
    ax.text(i+1, v+0.2, f"{v:.1f}%", ha="center", fontsize=10)

# 2b ─ Previous chargebacks
ax = axes[0, 1]
cb_fraud = df.groupby("prev_chargebacks")["is_fraud"].mean() * 100
ax.bar(cb_fraud.index, cb_fraud.values,
       color=["#E63946" if v > 10 else "#457B9D" for v in cb_fraud.values],
       edgecolor="white", width=0.6)
ax.set_xlabel("Previous Chargebacks"); ax.set_ylabel("Fraud Rate (%)")
ax.set_title("Chargeback History vs Fraud", fontweight="bold")
for i, v in zip(cb_fraud.index, cb_fraud.values):
    ax.text(i, v+0.5, f"{v:.1f}%", ha="center", fontsize=10)

# 2c ─ Account age vs fraud (boxplot)
ax = axes[0, 2]
df["fraud_label"] = df["is_fraud"].map({0: "Legit", 1: "Fraud"})
sns.boxplot(data=df, x="fraud_label", y="account_age_days", palette=palette, ax=ax,
            order=["Legit","Fraud"], width=0.5, linewidth=1.5)
ax.set_xlabel(""); ax.set_ylabel("Account Age (Days)")
ax.set_title("Account Age — Fraud vs Legit", fontweight="bold")

# 2d ─ Payment method fraud rate
ax = axes[1, 0]
pay_fraud = df.groupby("payment_method")["is_fraud"].mean().sort_values(ascending=False) * 100
ax.barh(pay_fraud.index, pay_fraud.values,
        color=["#E63946" if v > pay_fraud.mean() else "#457B9D" for v in pay_fraud.values],
        edgecolor="white")
ax.set_xlabel("Fraud Rate (%)"); ax.set_title("Fraud Rate by Payment Method", fontweight="bold")
for i, v in enumerate(pay_fraud.values):
    ax.text(v+0.1, i, f"{v:.1f}%", va="center", fontsize=10)

# 2e ─ Device type
ax = axes[1, 1]
dev_fraud = df.groupby("device_type")["is_fraud"].mean().sort_values(ascending=False) * 100
colors_d = ["#E63946","#F4A261","#2A9D8F"]
ax.bar(dev_fraud.index, dev_fraud.values, color=colors_d[:len(dev_fraud)], edgecolor="white", width=0.6)
ax.set_xlabel("Device Type"); ax.set_ylabel("Fraud Rate (%)")
ax.set_title("Fraud Rate by Device", fontweight="bold")
for i, v in enumerate(dev_fraud.values):
    ax.text(i, v+0.1, f"{v:.1f}%", ha="center", fontsize=11, fontweight="bold")

# 2f ─ Failed attempts
ax = axes[1, 2]
fa_fraud = df.groupby("failed_attempts")["is_fraud"].mean() * 100
ax.plot(fa_fraud.index, fa_fraud.values, marker="o", color="#E63946",
        linewidth=2.5, markersize=9, markerfacecolor="white", markeredgewidth=2.5)
ax.fill_between(fa_fraud.index, fa_fraud.values, alpha=0.15, color="#E63946")
ax.set_xlabel("Failed Attempts Before Success"); ax.set_ylabel("Fraud Rate (%)")
ax.set_title("Failed Attempts → Fraud Risk", fontweight="bold")
for x, y in zip(fa_fraud.index, fa_fraud.values):
    ax.text(x, y+0.5, f"{y:.1f}%", ha="center", fontsize=10)

plt.tight_layout()
fig.savefig("outputs/plots/02_risk_signals.png", dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.close()
print("   ✅ Plot 2 saved: 02_risk_signals.png")

# ─────────────────────────────────────────────────────────────
# PLOT 3 ─ Correlation Heatmap
# ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 10))
fig.patch.set_facecolor("#F8F9FA")
num_cols = ["amount","account_age_days","velocity_1h","prev_chargebacks",
            "failed_attempts","is_fraud"]
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdYlGn_r",
            ax=ax, linewidths=0.5, vmin=-1, vmax=1,
            cbar_kws={"shrink": 0.8})
ax.set_title("Feature Correlation Heatmap", fontsize=16, fontweight="bold", pad=15, color="#1D3557")
plt.tight_layout()
fig.savefig("outputs/plots/03_correlation.png", dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.close()
print("   ✅ Plot 3 saved: 03_correlation.png")

print("\n📊 EDA Summary Statistics:")
print(f"   Avg fraud amount   : ₹{fraud['amount'].mean():>10,.2f}")
print(f"   Avg legit amount   : ₹{legit['amount'].mean():>10,.2f}")
print(f"   Peak fraud hour    : {fraud['hour'].mode()[0]}:00")
print(f"   Top fraud country  : {fraud['country'].value_counts().index[0]}")
print(f"   High velocity fraud: {(fraud['velocity_1h']>=4).mean()*100:.1f}% of fraud txns have velocity ≥ 4")
print("✅ EDA complete!\n")

📊 Running Exploratory Data Analysis...
   ✅ Plot 1 saved: 01_eda_overview.png
   ✅ Plot 2 saved: 02_risk_signals.png
   ✅ Plot 3 saved: 03_correlation.png

📊 EDA Summary Statistics:
   Avg fraud amount   : ₹  2,245.78
   Avg legit amount   : ₹  1,419.46
   Peak fraud hour    : 21:00
   Top fraud country  : India
   High velocity fraud: 40.2% of fraud txns have velocity ≥ 4
✅ EDA complete!



In [5]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, precision_recall_curve,
                             average_precision_score, f1_score)
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
import joblib, os, warnings
warnings.filterwarnings("ignore")

os.makedirs("outputs/plots", exist_ok=True)
os.makedirs("models", exist_ok=True)

print("⚙️  Feature Engineering + Model Training...")

# ── Load data ──────────────────────────────────────────────────
df = pd.read_csv("data/transactions.csv", parse_dates=["timestamp"])

# ── Feature Engineering ────────────────────────────────────────
print("   [1/5] Engineering features...")

# Time features
df["hour"]           = df["timestamp"].dt.hour
df["is_night"]       = df["hour"].between(0, 5).astype(int)          # 12AM–5AM flag
df["is_weekend"]     = df["timestamp"].dt.dayofweek.isin([5,6]).astype(int)
df["month"]          = df["timestamp"].dt.month

# Amount features
user_avg_amount      = df.groupby("user_id")["amount"].transform("mean")
df["amount_vs_user_avg"] = df["amount"] / (user_avg_amount + 1)       # deviation from own avg
df["is_round_amount"]    = (df["amount"] % 100 == 0).astype(int)      # suspiciously round
df["amount_log"]         = np.log1p(df["amount"])

# Merchant risk score (historical fraud rate per merchant)
merch_fraud_rate     = df.groupby("merchant_id")["is_fraud"].transform("mean")
df["merchant_risk"]  = merch_fraud_rate

# User risk score
user_fraud_rate      = df.groupby("user_id")["is_fraud"].transform("mean")
df["user_risk"]      = user_fraud_rate

# Binary flags
df["ip_mismatch_flag"]   = (~df["ip_country_match"]).astype(int)
df["device_mismatch_flag"] = df["device_mismatch"].astype(int)
df["address_mismatch_flag"] = df["address_mismatch"].astype(int)

# Composite risk indicator (rule-based score)
df["rule_risk_score"] = (
    df["ip_mismatch_flag"] * 2 +
    df["device_mismatch_flag"] * 2 +
    df["address_mismatch_flag"] * 1 +
    (df["velocity_1h"] >= 4).astype(int) * 3 +
    (df["prev_chargebacks"] >= 1).astype(int) * 4 +
    (df["failed_attempts"] >= 2).astype(int) * 2 +
    (df["account_age_days"] < 30).astype(int) * 2 +
    df["is_night"] * 1 +
    (df["browser"] == "Unknown").astype(int) * 1
)

# ── Encode categoricals ────────────────────────────────────────
cat_cols = ["product_category","payment_method","device_type","browser","country","day_of_week"]
le = LabelEncoder()
for col in cat_cols:
    df[col + "_enc"] = le.fit_transform(df[col].astype(str))

# ── Feature matrix ──────────────────────────────────────────────
FEATURES = [
    "amount_log","amount_vs_user_avg","is_round_amount",
    "account_age_days","velocity_1h","prev_chargebacks","failed_attempts",
    "ip_mismatch_flag","device_mismatch_flag","address_mismatch_flag",
    "is_night","is_weekend","merchant_risk","user_risk","rule_risk_score",
    "product_category_enc","payment_method_enc","device_type_enc",
    "browser_enc","country_enc","hour","month"
]

X = df[FEATURES]
y = df["is_fraud"]

print(f"   [2/5] Dataset — X: {X.shape}, Fraud: {y.sum():,} ({y.mean()*100:.1f}%)")

# ── Train / Test split ──────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# ── SMOTE oversampling (handle class imbalance) ──────────────────
print("   [3/5] Applying SMOTE to balance classes...")
smote = SMOTE(random_state=42, k_neighbors=5)
X_res, y_res = smote.fit_resample(X_train, y_train)
print(f"          After SMOTE — Fraud: {y_res.sum():,}, Legit: {(1-y_res).sum():,}")

# ── Train models ────────────────────────────────────────────────
print("   [4/5] Training models...")

models = {
    "Random Forest":     RandomForestClassifier(n_estimators=200, max_depth=12,
                                                 class_weight="balanced", random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=150, max_depth=5,
                                                     learning_rate=0.08, random_state=42),
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf",    LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42))
    ])
}

results = {}
for name, model in models.items():
    model.fit(X_res, y_res)
    y_pred  = model.predict(X_test)
    y_prob  = model.predict_proba(X_test)[:, 1]
    auc     = roc_auc_score(y_test, y_prob)
    ap      = average_precision_score(y_test, y_prob)
    f1      = f1_score(y_test, y_pred)
    results[name] = {"model": model, "y_pred": y_pred, "y_prob": y_prob,
                     "auc": auc, "ap": ap, "f1": f1}
    print(f"          {name:25s} | AUC: {auc:.4f} | AP: {ap:.4f} | F1: {f1:.4f}")

# Best model = Random Forest
best_name  = max(results, key=lambda k: results[k]["auc"])
best       = results[best_name]
print(f"\n   🏆 Best model: {best_name} (AUC = {best['auc']:.4f})")

# Save model + feature list
joblib.dump(best["model"], "models/fraud_model.pkl")
pd.Series(FEATURES).to_csv("models/feature_list.csv", index=False)
print("   💾 Model saved to models/fraud_model.pkl")

# ── Risk Score: map probability → 0-100 ────────────────────────
df_test = X_test.copy()
df_test["fraud_prob"]  = best["y_prob"]
df_test["risk_score"]  = (best["y_prob"] * 100).round(1)
df_test["risk_label"]  = pd.cut(df_test["risk_score"],
                                  bins=[0, 30, 60, 80, 100],
                                  labels=["Low","Medium","High","Critical"])
df_test["is_fraud"]    = y_test.values
df_test["transaction_id"] = df.loc[X_test.index, "transaction_id"].values
df_test.to_csv("data/test_with_scores.csv", index=False)
print("   💾 Risk scores saved to data/test_with_scores.csv")

# ── PLOTS ───────────────────────────────────────────────────────
print("   [5/5] Generating model evaluation plots...")
sns.set_theme(style="whitegrid", font_scale=1.1)

fig = plt.figure(figsize=(22, 16))
fig.patch.set_facecolor("#F8F9FA")
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# Plot A ─ ROC curves for all models
ax = fig.add_subplot(gs[0, 0])
colors_roc = ["#E63946","#2A9D8F","#F4A261"]
for (name, res), c in zip(results.items(), colors_roc):
    fpr, tpr, _ = roc_curve(y_test, res["y_prob"])
    ax.plot(fpr, tpr, color=c, lw=2.5, label=f"{name} (AUC={res['auc']:.3f})")
ax.plot([0,1],[0,1],"k--", lw=1.2, alpha=0.5, label="Random")
ax.fill_between(*roc_curve(y_test, best["y_prob"])[:2], alpha=0.08, color="#E63946")
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves — All Models", fontweight="bold")
ax.legend(fontsize=9, loc="lower right")

# Plot B ─ Precision-Recall curve
ax = fig.add_subplot(gs[0, 1])
for (name, res), c in zip(results.items(), colors_roc):
    prec, rec, _ = precision_recall_curve(y_test, res["y_prob"])
    ax.plot(rec, prec, color=c, lw=2.5, label=f"{name} (AP={res['ap']:.3f})")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curves", fontweight="bold")
ax.legend(fontsize=9)

# Plot C ─ Confusion Matrix (best model)
ax = fig.add_subplot(gs[0, 2])
cm = confusion_matrix(y_test, best["y_pred"])
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
            xticklabels=["Legit","Fraud"], yticklabels=["Legit","Fraud"],
            linewidths=1, linecolor="white", annot_kws={"size": 14, "weight": "bold"})
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title(f"Confusion Matrix\n({best_name})", fontweight="bold")

# Plot D ─ Feature Importance
ax = fig.add_subplot(gs[1, :2])
rf_model = results["Random Forest"]["model"]
importances = pd.Series(rf_model.feature_importances_, index=FEATURES).sort_values(ascending=True)
top_n = importances.tail(15)
colors_fi = ["#E63946" if v > importances.median() else "#457B9D" for v in top_n.values]
top_n.plot(kind="barh", ax=ax, color=colors_fi, edgecolor="white")
ax.set_xlabel("Feature Importance Score")
ax.set_title("Top 15 Feature Importances (Random Forest)", fontweight="bold")
for i, v in enumerate(top_n.values):
    ax.text(v+0.001, i, f"{v:.3f}", va="center", fontsize=9)

# Plot E ─ Risk Score Distribution
ax = fig.add_subplot(gs[1, 2])
risk_colors = {"Low":"#2A9D8F","Medium":"#F4A261","High":"#E76F51","Critical":"#E63946"}
risk_counts = df_test["risk_label"].value_counts().reindex(["Low","Medium","High","Critical"])
bars = ax.bar(risk_counts.index, risk_counts.values,
              color=[risk_colors[r] for r in risk_counts.index],
              edgecolor="white", linewidth=1.5, width=0.6)
for bar, val in zip(bars, risk_counts.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+10,
            f"{val:,}", ha="center", fontsize=11, fontweight="bold")
ax.set_ylabel("Number of Transactions")
ax.set_title("Risk Score Distribution", fontweight="bold")

plt.suptitle(f"Model Evaluation Dashboard — Best: {best_name} (AUC={best['auc']:.4f})",
             fontsize=16, fontweight="bold", color="#1D3557", y=1.01)
fig.savefig("outputs/plots/04_model_evaluation.png", dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.close()
print("   ✅ Plot saved: 04_model_evaluation.png")

# Classification report
print(f"\n📋 Classification Report — {best_name}:")
print(classification_report(y_test, best["y_pred"], target_names=["Legit","Fraud"]))
print("✅ Model training complete!\n")

⚙️  Feature Engineering + Model Training...
   [1/5] Engineering features...
   [2/5] Dataset — X: (50000, 22), Fraud: 2,775 (5.5%)
   [3/5] Applying SMOTE to balance classes...
          After SMOTE — Fraud: 37,780, Legit: 37,780
   [4/5] Training models...
          Random Forest             | AUC: 0.9749 | AP: 0.7559 | F1: 0.6450
          Gradient Boosting         | AUC: 0.9800 | AP: 0.8588 | F1: 0.7531
          Logistic Regression       | AUC: 0.9503 | AP: 0.6193 | F1: 0.5505

   🏆 Best model: Gradient Boosting (AUC = 0.9800)
   💾 Model saved to models/fraud_model.pkl
   💾 Risk scores saved to data/test_with_scores.csv
   [5/5] Generating model evaluation plots...
   ✅ Plot saved: 04_model_evaluation.png

📋 Classification Report — Gradient Boosting:
              precision    recall  f1-score   support

       Legit       0.99      0.98      0.98      9445
       Fraud       0.69      0.83      0.75       555

    accuracy                           0.97     10000
   macro avg    

In [12]:
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import seaborn as sns
import os, warnings
warnings.filterwarnings("ignore")

os.makedirs("outputs/plots", exist_ok=True)
os.makedirs("outputs/reports", exist_ok=True)

df = pd.read_csv("data/transactions.csv")
df["timestamp"] = pd.to_datetime(df["timestamp"], format='mixed')
df_score = pd.read_csv("data/test_with_scores.csv")
fraud    = df[df["is_fraud"] == 1]
legit    = df[df["is_fraud"] == 0]

sns.set_theme(style="white", font_scale=1.0)

# ═══════════════════════════════════════════════════════════════
# DASHBOARD 1 ─ EXECUTIVE SUMMARY DASHBOARD
# ═══════════════════════════════════════════════════════════════
fig = plt.figure(figsize=(24, 16))
fig.patch.set_facecolor("#0F172A")   # dark navy background

# ── Title banner ────────────────────────────────────────────────
fig.text(0.5, 0.97, "🛡️  E-COMMERCE FRAUD DETECTION DASHBOARD",
         ha="center", va="top", fontsize=22, fontweight="bold",
         color="white", fontfamily="monospace")
fig.text(0.5, 0.94, "Real-Time Risk Monitoring  |  50,000 Transactions  |  2023",
         ha="center", va="top", fontsize=13, color="#94A3B8")

# ── KPI Cards (top row) ─────────────────────────────────────────
kpis = [
    ("💳  Total Transactions",  f"50,000",                     "#1E3A5F", "#60A5FA"),
    ("🚨  Fraud Transactions",  f"2,775",                       "#4A1942", "#F472B6"),
    ("📉  Fraud Rate",          f"5.5%",                        "#1A3A2A", "#4ADE80"),
    ("💸  Total Fraud Loss",    f"₹62,32,035",                  "#4A2A1A", "#FB923C"),
    ("⚡  Avg Fraud Amount",    f"₹2,245",                      "#2A1A4A", "#A78BFA"),
    ("🔒  Model AUC Score",     f"0.9800",                      "#1A3A3A", "#22D3EE"),
]
kpi_axes_positions = [(0.01+i*0.165, 0.78, 0.155, 0.13) for i in range(6)]
for (title, value, bg, fg), pos in zip(kpis, kpi_axes_positions):
    ax_kpi = fig.add_axes(pos)
    ax_kpi.set_facecolor(bg)
    ax_kpi.set_xlim(0, 1); ax_kpi.set_ylim(0, 1)
    ax_kpi.axis("off")
    ax_kpi.text(0.5, 0.72, value, ha="center", va="center",
                fontsize=22, fontweight="bold", color=fg)
    ax_kpi.text(0.5, 0.25, title, ha="center", va="center",
                fontsize=9.5, color="#CBD5E1", wrap=True)
    for spine in ["top","right","bottom","left"]:
        ax_kpi.spines[spine].set_visible(True)
        ax_kpi.spines[spine].set_color(fg)
        ax_kpi.spines[spine].set_linewidth(1.5)

gs = gridspec.GridSpec(2, 3, figure=fig,
                       left=0.04, right=0.98, top=0.75, bottom=0.06,
                       hspace=0.45, wspace=0.28)

DARK = "#0F172A"; CARD = "#1E293B"; TEXT = "white"; MUTED = "#94A3B8"

def dark_ax(ax):
    ax.set_facecolor(CARD)
    ax.tick_params(colors=MUTED, labelsize=9)
    ax.xaxis.label.set_color(MUTED); ax.yaxis.label.set_color(MUTED)
    ax.title.set_color(TEXT)
    for spine in ax.spines.values(): spine.set_color("#334155")

# ── Plot 1: Monthly Fraud Trend ──────────────────────────────────
ax1 = fig.add_subplot(gs[0, :2])
dark_ax(ax1)
monthly = df.groupby("month").agg(
    total=("is_fraud","count"), fraud=("is_fraud","sum"),
    fraud_amount=("amount", lambda x: x[df.loc[x.index,"is_fraud"]==1].sum())
).reset_index()
months_label = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
ax1.plot(monthly["month"], monthly["fraud"], color="#F472B6",
         lw=2.5, marker="o", markersize=7, markerfacecolor="#0F172A",
         markeredgewidth=2, label="Fraud Count")
ax1b = ax1.twinx()
dark_ax(ax1b)
ax1b.fill_between(monthly["month"], monthly["fraud_amount"]/1e5, alpha=0.2, color="#60A5FA")
ax1b.plot(monthly["month"], monthly["fraud_amount"]/1e5, color="#60A5FA",
          lw=1.5, linestyle="--", label="Fraud Amount (₹L)")
ax1.set_xticks(range(1,13)); ax1.set_xticklabels(months_label, color=MUTED)
ax1.set_ylabel("Fraud Transactions", color="#F472B6")
ax1b.set_ylabel("Fraud Amount (₹ Lakhs)", color="#60A5FA")
ax1b.tick_params(colors=MUTED, labelsize=9)
ax1.set_title("📈 Monthly Fraud Trend — Count & Amount", fontweight="bold", color=TEXT, pad=10)
lines1, labs1 = ax1.get_legend_handles_labels()
lines2, labs2 = ax1b.get_legend_handles_labels()
ax1.legend(lines1+lines2, labs1+labs2, loc="upper right", fontsize=9,
           facecolor=CARD, labelcolor=MUTED, edgecolor="#334155")

# ── Plot 2: Risk Score Donut ──────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 2])
dark_ax(ax2); ax2.set_facecolor(DARK)
risk_counts = df_score["risk_label"].value_counts().reindex(["Low","Medium","High","Critical"]).fillna(0)
colors_risk = ["#22D3EE","#F4A261","#F97316","#EF4444"]
wedges, texts, autotexts = ax2.pie(
    risk_counts, labels=risk_counts.index, autopct="%1.1f%%",
    colors=colors_risk, startangle=90,
    pctdistance=0.75, wedgeprops=dict(width=0.55, edgecolor=DARK, linewidth=2),
    textprops={"color": MUTED, "fontsize": 9}
)
for at in autotexts: at.set_color(TEXT); at.set_fontweight("bold"); at.set_fontsize(10)
ax2.set_title("🎯 Risk Score Distribution", fontweight="bold", color=TEXT, pad=10)

# ── Plot 3: Fraud by Country (bar) ───────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
dark_ax(ax3)
ct = df.groupby("country")["is_fraud"].mean().sort_values(ascending=True) * 100
bar_colors = ["#EF4444" if v > ct.mean() else "#3B82F6" for v in ct.values]
bars = ax3.barh(ct.index, ct.values, color=bar_colors, edgecolor=DARK, linewidth=0.5)
for bar, v in zip(bars, ct.values):
    ax3.text(v+0.05, bar.get_y()+bar.get_height()/2, f"{v:.1f}%",
             va="center", fontsize=9, color=TEXT)
ax3.set_xlabel("Fraud Rate (%)", color=MUTED)
ax3.set_title("🌍 Fraud Rate by Country", fontweight="bold", color=TEXT, pad=10)
ax3.axvline(ct.mean(), color="#F472B6", linestyle="--", lw=1.2, alpha=0.7)
ax3.text(ct.mean()+0.05, -0.6, f"Avg: {ct.mean():.1f}%", fontsize=8, color="#F472B6")

# ── Plot 4: Heatmap ────────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
dark_ax(ax4)
day_order = ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"]
df["dow_short"] = df["timestamp"].dt.strftime("%a")
hm = df[df["is_fraud"]==1].groupby(["dow_short","hour"]).size().unstack(fill_value=0)
hm = hm.reindex([d for d in day_order if d in hm.index])
sns.heatmap(hm, ax=ax4, cmap="YlOrRd", linewidths=0.2, linecolor="#0F172A",
            cbar_kws={"shrink":0.8, "label":"Fraud Count"})
ax4.set_title("🔥 Fraud Heatmap (Day × Hour)", fontweight="bold", color=TEXT, pad=10)
ax4.set_xlabel("Hour of Day", color=MUTED); ax4.set_ylabel("", color=MUTED)
ax4.collections[0].colorbar.ax.yaxis.set_tick_params(color=MUTED)
ax4.collections[0].colorbar.ax.tick_params(labelcolor=MUTED)

# ── Plot 5: Risk Signal Summary ──────────────────────────────────
ax5 = fig.add_subplot(gs[1, 2])
dark_ax(ax5)
signals = {
    "IP Mismatch":         df[df["is_fraud"]==1]["ip_country_match"].eq(False).mean()*100,
    "Device Mismatch":     df[df["is_fraud"]==1]["device_mismatch"].mean()*100,
    "Address Mismatch":    df[df["is_fraud"]==1]["address_mismatch"].mean()*100,
    "High Velocity (≥4)":  (df[df["is_fraud"]==1]["velocity_1h"]>=4).mean()*100,
    "Failed Attempts ≥2":  (df[df["is_fraud"]==1]["failed_attempts"]>=2).mean()*100,
    "Prior Chargeback":    (df[df["is_fraud"]==1]["prev_chargebacks"]>=1).mean()*100,
    "New Account (<30d)":  (df[df["is_fraud"]==1]["account_age_days"]<30).mean()*100,
}
sig_df = pd.Series(signals).sort_values()
colors_sig = plt.cm.RdYlGn_r(np.linspace(0.2, 0.9, len(sig_df)))
bars5 = ax5.barh(sig_df.index, sig_df.values, color=colors_sig, edgecolor=DARK)
for bar, v in zip(bars5, sig_df.values):
    ax5.text(v+0.5, bar.get_y()+bar.get_height()/2, f"{v:.1f}%",
             va="center", fontsize=9, color=TEXT)
ax5.set_xlabel("% of Fraud Transactions with Signal", color=MUTED)
ax5.set_title("⚡ Risk Signal Prevalence in Fraud", fontweight="bold", color=TEXT, pad=10)

fig.savefig("outputs/plots/05_executive_dashboard.png", dpi=150,
            bbox_inches="tight", facecolor=fig.get_facecolor())
plt.close()
print("✅ Dashboard saved: 05_executive_dashboard.png")

# ════════════════════════════════════════════════════════════════
# DASHBOARD 2 ─ MERCHANT RISK SCORECARD
# ════════════════════════════════════════════════════════════════
import duckdb
con = duckdb.connect()
con.register("transactions", df)

merchant_risk = con.execute("""
    SELECT merchant_id,
           COUNT(*) total_txns,
           SUM(is_fraud) fraud_txns,
           ROUND(AVG(is_fraud)*100,2) fraud_rate_pct,
           ROUND(SUM(CASE WHEN is_fraud=1 THEN amount ELSE 0 END),2) revenue_at_risk
    FROM transactions
    GROUP BY merchant_id
    HAVING COUNT(*) >= 50
    ORDER BY fraud_rate_pct DESC
    LIMIT 20
""").df()

fig, axes = plt.subplots(1, 2, figsize=(20, 9))
fig.patch.set_facecolor("#0F172A")
fig.suptitle("🏪 Merchant Risk Scorecard — Top 20 Risky Merchants",
             fontsize=16, fontweight="bold", color="white", y=1.01)

ax = axes[0]
ax.set_facecolor("#1E293B")
colors_m = ["#EF4444" if r > 7 else "#F97316" if r > 5 else "#22D3EE"
            for r in merchant_risk["fraud_rate_pct"]]
bars = ax.barh(merchant_risk["merchant_id"], merchant_risk["fraud_rate_pct"],
               color=colors_m, edgecolor="#0F172A")
ax.set_xlabel("Fraud Rate (%)", color="#94A3B8")
ax.set_title("Fraud Rate by Merchant", fontweight="bold", color="white")
ax.tick_params(colors="#94A3B8")
for spine in ax.spines.values(): spine.set_color("#334155")
for bar, v in zip(bars, merchant_risk["fraud_rate_pct"]):
    ax.text(v+0.05, bar.get_y()+bar.get_height()/2, f"{v}%",
            va="center", fontsize=8, color="white")
patches = [mpatches.Patch(color="#EF4444", label="High Risk (>7%)"),
           mpatches.Patch(color="#F97316", label="Medium Risk (5-7%)"),
           mpatches.Patch(color="#22D3EE", label="Low Risk (<5%)")]
ax.legend(handles=patches, facecolor="#1E293B", labelcolor="#94A3B8",
          edgecolor="#334155", fontsize=9)

ax2 = axes[1]
ax2.set_facecolor("#1E293B")
ax2.scatter(merchant_risk["total_txns"], merchant_risk["revenue_at_risk"]/1000,
            c=merchant_risk["fraud_rate_pct"], cmap="RdYlGn_r",
            s=merchant_risk["fraud_txns"]*8, alpha=0.8, edgecolors="white", linewidth=0.5)
for _, row in merchant_risk.head(5).iterrows():
    ax2.annotate(row["merchant_id"], (row["total_txns"], row["revenue_at_risk"]/1000),
                 textcoords="offset points", xytext=(5, 5),
                 fontsize=8, color="white")
ax2.set_xlabel("Total Transactions", color="#94A3B8")
ax2.set_ylabel("Revenue at Risk (₹ Thousands)", color="#94A3B8")
ax2.set_title("Transaction Volume vs Revenue at Risk\n(bubble size = fraud count)",
              fontweight="bold", color="white")
ax2.tick_params(colors="#94A3B8")
for spine in ax2.spines.values(): spine.set_color("#334155")

plt.tight_layout()
fig.savefig("outputs/plots/06_merchant_scorecard.png", dpi=150,
            bbox_inches="tight", facecolor=fig.get_facecolor())
plt.close()
print("✅ Dashboard saved: 06_merchant_scorecard.png")

print("\n🎉 All dashboards generated successfully!")

✅ Dashboard saved: 05_executive_dashboard.png
✅ Dashboard saved: 06_merchant_scorecard.png

🎉 All dashboards generated successfully!
